In [0]:
import requests
import json
from datetime import datetime, timezone, timedelta

In [0]:
base_url = "https://hapi.fhir.org/baseR4/Patient"


In [0]:
end_date = datetime.now(timezone.utc).replace(microsecond=0)
start_date = end_date - timedelta(days=3)

In [0]:
start_date = start_date.isoformat().replace("+00:00", "Z")
end_date = end_date.isoformat().replace("+00:00", "Z")

In [0]:
params = [
    ("_lastUpdated", "ge" + start_date),
    ("_lastUpdated", "lt" + end_date),
    ("_count", "100")
]

In [0]:
url = base_url
page = 1
total = 0

while url:

    if page == 1:
        result = requests.get(
            url,
            params=params,
            headers={"Accept": "application/fhir+json"},
            timeout=60
        )
    else:
        result = requests.get(
            url,
            headers={"Accept": "application/fhir+json"},
            timeout=60
        )

    print("Status:", result.status_code)

    result.raise_for_status()

    data = result.json()
    records = data.get("entry", [])

    total = total + len(records)

    today = datetime.now(timezone.utc).strftime("%Y-%m-%d")
    folder = f"dbfs:/Volumes/workspace/default/fhir_files/raw/patient/extraction_date={today}"

    dbutils.fs.mkdirs(folder)

    file_name = f"{folder}/patient_page_{page}.json"

    dbutils.fs.put(
        file_name,
        result.text,
        True
    )

    print("Page", page, "saved")
    print("Records:", len(records))
    print("File:", file_name)

    url = None

    for link in data.get("link", []):
        if link.get("relation") == "next":
            url = link.get("url")

    page = page + 1

print("Total records:", total)
print("Data fetched from:", start_date)
print("Data fetched until:", end_date)

In [0]:
# display(dbutils.fs.ls("dbfs:/Workspace/fhir_assignment/"))

In [0]:
data